In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np


In [49]:
con = sqlite3.connect("../vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "../v33.db" AS v33')

## Create necessary table

In [28]:
# base table with root counts

cur.execute("""
drop table if exists transaction_v2_obl_root_eluskoht_counts_base
""")

cur.execute("""
create table transaction_v2_obl_root_eluskoht_counts_base as
SELECT lemma, count(lemma) as lemmacnt
from 
transaction_v2
where deprel = 'obl'
group by lemma
""")

In [29]:
# table counts elus 

cur.execute("""
drop table if exists transaction_v2_obl_root_eluskoht_counts_elus
""")

cur.execute("""
Create table transaction_v2_obl_root_eluskoht_counts_elus as
SELECT lemma, count(lemma) as elus_cnt
FROM 
transaction_v2
where deprel = 'obl'
and elus = 'YES'
group by lemma
""")

In [30]:
# tbl count koht

cur.execute("""
drop table if exists transaction_v2_obl_root_eluskoht_counts_koht
""")

cur.execute("""
Create table transaction_v2_obl_root_eluskoht_counts_koht as
SELECT lemma, count(*) as koht_cnt
FROM 
transaction_v2
where deprel = 'obl'
and koht = 'YES'
group by lemma
""")

In [31]:
# join everythin g into 1 table

cur.execute("""DROP table if exists transaction_v2_obl_root_eluskoht_counts_1""")
cur.execute("""
CREATE TABLE transaction_v2_obl_root_eluskoht_counts_1 AS
select tbl1.lemma, tbl1.lemmacnt as lemma_cnt, elus_cnt
from
transaction_v2_obl_root_eluskoht_counts_base as tbl1
left join
transaction_v2_obl_root_eluskoht_counts_elus as tbl2
on tbl1.lemma=tbl2.lemma 
""")


cur.execute("""DROP table if exists transaction_v2_obl_root_eluskoht_counts""")
cur.execute("""
CREATE TABLE transaction_v2_obl_root_eluskoht_counts AS
select tbl1.lemma, tbl1.lemma_cnt, tbl1.elus_cnt, koht_cnt
from
transaction_v2_obl_root_eluskoht_counts_1 as tbl1
left join
transaction_v2_obl_root_eluskoht_counts_koht as tbl2
on tbl1.lemma=tbl2.lemma 
""")

In [32]:
# update table null -> 0

cur.execute("""
UPDATE transaction_v2_obl_root_eluskoht_counts
SET elus_cnt = 0
where elus_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transaction_v2_obl_root_eluskoht_counts
SET koht_cnt = 0
where koht_cnt is null
""")
con.commit()

In [47]:
query = """
SELECT * 
from 
transaction_v2_obl_root_eluskoht_counts
where elus_cnt!=koht_cnt
limit 20
"""

source2 = pd.read_sql_query(query, con)
source2

,lemma,lemma_cnt,elus_cnt,koht_cnt
0,2toaline,1,0,1
1,3toaline,1,0,1
2,AAV,1,0,1
3,Aadress,1,0,1
4,Aafrika,1168,0,1168
5,Abikaasa,2,2,0
6,Abilinnapea,1,1,0
7,Afgaan,1,1,0
8,Agent,2,2,0
9,Ajakirjanik,2,2,0


In [51]:
query = """
SELECT * 
from 
transaction_v2_obl_root_eluskoht_counts
where elus_cnt!=lemma_cnt and elus_cnt>0
limit 20
"""

source2 = pd.read_sql_query(query, con)
source2

,lemma,lemma_cnt,elus_cnt,koht_cnt


In [52]:
con.close()